### Importación de librerías

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from matplotlib.colors import ListedColormap

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)

from typing import List, Optional, Tuple, Union

In [2]:
PATH_DIRECTORIO_DATOS = "../../data"
PATH_DATASET_PRACTICA_FINAL = f"{PATH_DIRECTORIO_DATOS}/processed/dataset_practica_final_preprocessed.csv"

# Cargamos el dataset de la práctica final
df_preprocessed = pd.read_csv(PATH_DATASET_PRACTICA_FINAL)

In [3]:
# Quitamos nulos
# df_preprocessed = df_preprocessed.dropna()

# Quitamos duplicados
# df_preprocessed = df_preprocessed.drop_duplicates()

In [4]:
df_preprocessed.shape

(119390, 46)

In [5]:
df_preprocessed.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 46 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   lead_time                       119390 non-null  float64
 1   stays_in_weekend_nights         119390 non-null  float64
 2   stays_in_week_nights            119390 non-null  float64
 3   adults                          119390 non-null  float64
 4   children                        119386 non-null  float64
 5   babies                          119390 non-null  float64
 6   is_repeated_guest               119390 non-null  float64
 7   previous_cancellations          119390 non-null  float64
 8   previous_bookings_not_canceled  119390 non-null  float64
 9   booking_changes                 119390 non-null  float64
 10  agent                           103050 non-null  float64
 11  company                         6797 non-null    float64
 12  adr                        

In [6]:
# Contar valores nulos
df_preprocessed.isna().sum()

lead_time                              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
booking_changes                        0
agent                              16340
company                           112593
adr                                    0
hotel_city hotel                       0
hotel_resort hotel                     0
market_segment_aviation                0
market_segment_complementary           0
market_segment_corporate               0
market_segment_direct                  0
market_segment_groups                  0
market_segment_offline ta/to           0
market_segment_online ta               0
market_segment_undefined               0
distribution_channel_corporate         0
distribution_cha

In [7]:
# Verificar si hay valores duplicados en el dataframe
df_preprocessed.duplicated().sum()

np.int64(38124)

# Modelo Regresión Logística
---

In [8]:
target_column = 'is_canceled'

# Reemplazamos los valores de texto de la columna 'target' por los valores numéricos de las clases
dict_target = {
    'No cancelado': 0,
    'Cancelado': 1
}

#df_preprocessed[target_column] = df_preprocessed[target_column].replace(dict_target)

In [9]:
# Preparación de los datos para el modelo de regresión logística
# Eliminamos la columna 'target' de las variables independientes
# Y transformamos a numérico la columna target
# X = df_preprocessed.drop(columns=target)
X = df_preprocessed.drop(columns=target_column)
y = df_preprocessed[target_column]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Comprobar proporción de datos entre test y train
len(X_test) / (len(X_train) + len(X_test))

0.2

## Verificación de ratio de clases (cancelado y no cancelado)

In [11]:
# Ratio de clases en la columna target (is_canceled) en el dataset
y.value_counts(normalize=True)

is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64

In [12]:
# Ratio de clases en la columna target en el conjunto de entrenamiento
y_train.value_counts(normalize=True)

is_canceled
0    0.630905
1    0.369095
Name: proportion, dtype: float64

In [ ]:
print(y_train)

67702     1
115851    0
57345     1
11622     1
33333     0
         ..
76820     0
110268    0
103694    0
860       1
15795     0
Name: is_canceled, Length: 95512, dtype: int64


In [ ]:
# Ratio de clases en la columna target en el conjunto de prueba
y_test.value_counts(normalize=True)

is_canceled
0    0.624299
1    0.375701
Name: proportion, dtype: float64

In [ ]:
# Se observa que el ratio difiere.
# Hacer train-test split utilizando stratify para que la proporción de clases se mantenga en ambos conjuntos
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Verificar el ratio de las clases para los conjuntos de entrenamiento y prueba
print("Distribución de clases en el conjunto de entrenamiento:")
print(y_train.value_counts(normalize=True))

print("\nDistribución de clases en el conjunto de prueba:")
print(y_test.value_counts(normalize=True))

Distribución de clases en el conjunto de entrenamiento:
is_canceled
0    0.630905
1    0.369095
Name: proportion, dtype: float64

Distribución de clases en el conjunto de prueba:
is_canceled
0    0.624299
1    0.375701
Name: proportion, dtype: float64


---

## Entrenamiento

**OJO: revisar cómo se escoge el valor "cv" cuando ejecuto GridSeachCV**

In [20]:
# Entrenamos un modelo usando GridSearchCV para encontrar los mejores hiperparámetros
dict_parametros = {
    'C': [0.01, 0.1, 1, 10],
    'max_iter' : [50, 100, 200],
    'solver': ['liblinear']
}

modelo_rl = LogisticRegression(random_state=42)
modelo_rl_cv = GridSearchCV(modelo_rl, dict_parametros, cv=5, scoring='accuracy')
modelo_rl_cv.fit(X_train, y_train)

ValueError: 
All the 60 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Jaime\Documents\Pontia\modulo-05-machine-learning\pontia-ml-proyecto-final\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Jaime\Documents\Pontia\modulo-05-machine-learning\pontia-ml-proyecto-final\.venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Jaime\Documents\Pontia\modulo-05-machine-learning\pontia-ml-proyecto-final\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py", line

In [ ]:
# Mostramos los mejores hiperparámetros encontrados
print(f"Mejores hiperparámetros encontrados: {modelo_rl_cv.best_params_}")
print(f"Mejor score obtenido: {modelo_rl_cv.best_score_:.2%}")

Mejores hiperparámetros encontrados: {'C': 0.1, 'max_iter': 50, 'solver': 'liblinear'}
Mejor score obtenido: 91.82%


In [ ]:
# Entrenamos el modelo con los mejores hiperparámetros encontrados
modelo_rl = LogisticRegression(C=0.1, max_iter=100, solver='liblinear', random_state=42)
modelo_rl.fit(X_train, y_train)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1

---

## Evaluación



In [ ]:
# Verificamos las clases del modelo
print(f"Clases del modelo: {modelo_rl.classes_}")

Clases del modelo: [0 1]


In [ ]:
print(X_test)

        lead_time  stays_in_weekend_nights  stays_in_week_nights    adults  \
41142   -0.973319                -0.928890             -0.262174  0.247897   
2988    -0.692585                 1.073895              1.309924  0.247897   
31325   -0.945245                -0.928890             -0.786207 -1.478447   
2983    -0.617722                 0.072502              1.309924 -1.478447   
81488   -0.935887                -0.928890             -0.262174  0.247897   
111424  -0.870383                -0.928890             -0.262174 -1.478447   
113046  -0.851667                 0.072502              0.261858 -1.478447   
3071    -0.954603                -0.928890             -0.786207  0.247897   
21079   -0.879741                 0.072502             -0.262174 -1.478447   
16645   -0.814236                 1.073895             -0.786207 -1.478447   
2916    -0.645795                 3.076680              2.357989 -1.478447   
103990   1.328702                 1.073895              1.309924

In [ ]:
# Hacemos predicciones sobre el conjunto de test
y_pred = modelo_rl.predict(X_test)

In [ ]:
y_pred

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])